# Бизнес-сообщества ВКонтакте: кто готов к социальной коммерции

Анализ `Business_groups.csv` за 30 дней сентября 2026 года. Таблица описывает сторону предложения: сообщества, контент, каталог, инструменты торговли, заказы и рекламу. Цель — выбрать приоритетные бизнесы для развития Маркета ВКонтакте и объяснить, что им мешает расти.

Данные синтетические. Все связи с таблицами Маркета ниже — связи на уровне тематик и продуктовых гипотез, а не идентификаторов одного и того же бизнеса или пользователя.

## Как читать анализ

В выборке есть товарные, сервисные и смешанные сообщества. Наличие чекаута не означает, что сообщество активно продаёт: отдельно проверяются каталог, товарный контент, открытия карточек, заказы и GMV.

Для выбора целевых бизнесов использованы три критерия:

1. есть товарное предложение или понятный путь его создания;
2. есть аудитория и контент, через которые можно приводить спрос;
3. есть измеримый барьер между интересом и заказом.

Это помогает разделить задачу подключения инструмента и задачу роста уже работающего продавца.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

data_path = Path.cwd().parent / 'Data'
pd.set_option('display.float_format', '{:,.2f}'.format)
groups = pd.read_csv(data_path / 'Business_groups.csv')
market = pd.read_csv(data_path / 'Market_Partner.csv').set_index('Метрика')
traffic = pd.read_csv(data_path / 'Market_Partner_traffic.csv').set_index('Источник')
categories = pd.read_csv(data_path / 'Market_Partner_categories.csv').set_index('Категория')

groups['Есть каталог'] = (
    groups['Товары загружены'].eq(1)
    | groups['Число активных товаров в каталоге'].gt(0)
)
groups['Есть товарный контент'] = (
    groups['Посты с товаром за 30 дней']
    + groups['Клипы с товаром за 30 дней']
).gt(0)
groups['Есть чекаут'] = groups['Подключён чекаут'].eq(1)
groups['Есть доставка'] = groups['Подключена доставка'].eq(1)
groups['Есть оплата'] = groups['Подключена оплата'].eq(1)
groups['Есть заказы'] = groups['Заказы через чекаут'].gt(0)
groups['Есть реклама'] = groups['Траты на рекламу всего, ₽'].gt(0)
groups['Engagement rate, %'] = (
    groups['Лайки обычного контента'] + groups['Репосты обычного контента']
) / groups['Просмотры обычного контента'].replace(0, np.nan) * 100
groups['Открытия на просмотр товарного контента, %'] = (
    groups['Открытия карточки товара из контента']
    / groups['Просмотры товарного контента'].replace(0, np.nan) * 100
)
groups['Заказы на открытие карточки, %'] = (
    groups['Заказы через чекаут']
    / groups['Открытия карточки товара из контента'].replace(0, np.nan) * 100
)
groups['GMV на заказ, ₽'] = (
    groups['GMV через чекаут, ₽']
    / groups['Заказы через чекаут'].replace(0, np.nan)
)
groups.shape

(9754, 39)

## 1. Общий профиль выборки

Выборка широкая: в ней много небольших сообществ без каталога и без чекаута. Поэтому главный барьер для большинства — не оптимизация уже существующей воронки, а переход от контента и интереса к структурированному товарному предложению. Отдельная группа уже подключила чекаут, но не получила заказов: для неё важнее диагностика карточки, ассортимента, доставки и оплаты.

In [2]:
overview = pd.DataFrame({
    'Значение': [
        len(groups),
        groups['Число подписчиков'].median(),
        groups['Есть каталог'].sum(),
        groups['Есть товарный контент'].sum(),
        groups['Есть чекаут'].sum(),
        groups['Есть заказы'].sum(),
        groups['GMV через чекаут, ₽'].sum(),
    ]
}, index=[
    'Сообщества',
    'Медиана подписчиков',
    'Есть каталог',
    'Есть товарный контент',
    'Есть чекаут',
    'Есть заказы через чекаут',
    'GMV через чекаут, ₽',
])
overview

,Значение
Сообщества,"9,754.00"
Медиана подписчиков,490.00
Есть каталог,"4,107.00"
Есть товарный контент,303.00
Есть чекаут,"5,737.00"
Есть заказы через чекаут,298.00
"GMV через чекаут, ₽","9,973,027.00"


In [3]:
type_summary = groups.groupby('Тип бизнеса (товары / услуги)').agg(
    Бизнесов=('ID бизнеса', 'size'),
    Каталог=('Есть каталог', 'mean'),
    Чекаут=('Есть чекаут', 'mean'),
    Заказы=('Заказы через чекаут', 'sum'),
    GMV=('GMV через чекаут, ₽', 'sum'),
).reset_index()
type_summary['Каталог, %'] = type_summary['Каталог'] * 100
type_summary['Чекаут, %'] = type_summary['Чекаут'] * 100
type_summary = type_summary.sort_values('Бизнесов', ascending=False)

fig = go.Figure()
fig.add_bar(
    x=type_summary['Тип бизнеса (товары / услуги)'],
    y=type_summary['Бизнесов'],
    name='Все сообщества',
    marker_color='#B8C2CC',
    hovertemplate='%{x}<br>Сообществ: %{y:,.0f}<extra></extra>',
)
fig.add_bar(
    x=type_summary['Тип бизнеса (товары / услуги)'],
    y=type_summary['Заказы'],
    name='Оформленные заказы',
    marker_color='#3978C6',
    yaxis='y2',
    hovertemplate='%{x}<br>Заказов: %{y:,.0f}<extra></extra>',
)
fig.update_layout(
    template='plotly_white',
    barmode='group',
    height=500,
    title='Размер типов бизнеса и фактические заказы через чекаут',
    xaxis_title='Тип бизнеса',
    yaxis=dict(title='Сообщества'),
    yaxis2=dict(title='Оформленные заказы', overlaying='y', side='right'),
    legend=dict(orientation='h', y=1.12),
    margin=dict(l=70, r=80, t=100, b=100),
)
fig.show(config={'displaylogo': False, 'responsive': True})

**Вывод.** Основной целевой массив — товарные и смешанные сообщества: они уже содержат продукт, каталог и потенциальную транзакционную модель. Сервисные и сообщества без каталога нельзя исключать, но для них нужен отдельный путь — заявки, переписка или услуги, а не товарный чекаут как универсальное решение.

## 2. Из каких сообществ складывается выборка

Тематическая структура показывает одновременно масштаб потенциальной базы и разную готовность к торговле. Для приоритизации важны не только крупные тематики, но и сочетание трёх показателей: доля сообществ с каталогом, доля с чекаутом и фактические заказы.

In [4]:
theme_summary = groups.groupby('Тематика').agg(
    Сообществ=('ID бизнеса', 'size'),
    Медиана_подписчиков=('Число подписчиков', 'median'),
    С_каталогом=('Есть каталог', 'mean'),
    С_чекаутом=('Есть чекаут', 'mean'),
    С_заказами=('Есть заказы', 'sum'),
    Открытия_карточек=('Открытия карточки товара из контента', 'sum'),
    Заказы=('Заказы через чекаут', 'sum'),
    GMV=('GMV через чекаут, ₽', 'sum'),
).reset_index()
theme_summary['Каталог, %'] = theme_summary['С_каталогом'] * 100
theme_summary['Чекаут, %'] = theme_summary['С_чекаутом'] * 100
theme_summary['Доля заказов, %'] = theme_summary['Заказы'] / theme_summary['Заказы'].sum() * 100
theme_summary = theme_summary.sort_values('Сообществ', ascending=False)
theme_summary.head(20)

,Тематика,Сообществ,Медиана_подписчиков,С_каталогом,С_чекаутом,С_заказами,Открытия_карточек,Заказы,GMV,"Каталог, %","Чекаут, %","Доля заказов, %"
9,Красота и здоровье,1703,310.00,0.26,0.48,13,1226,57,435010,26.01,48.39,1.64
10,Кулинария и еда,837,830.00,0.58,0.67,66,914,1187,2405730,57.71,66.79,34.13
4,Дом и ремонт,744,595.00,0.51,0.62,12,116,58,424193,50.94,62.50,1.67
33,Творчество и хендмейд,538,380.00,0.55,0.67,28,2263,339,291680,55.39,67.10,9.75
20,Образование,537,560.00,0.39,0.51,19,8147,337,631710,39.48,51.02,9.69
36,Транспорт,521,620.00,0.40,0.57,14,193,176,1229470,40.50,56.81,5.06
16,Мода и стиль,460,860.00,0.57,0.71,21,8949,154,583861,57.39,70.65,4.43
1,Бизнес и работа,436,360.00,0.38,0.55,6,294,21,16720,38.30,55.05,0.60
26,Путешествия и отдых,416,"1,200.00",0.49,0.61,8,2176,8,54640,48.56,60.82,0.23
35,Товары и магазины,405,350.00,0.65,0.80,40,1281,338,1587270,65.43,80.00,9.72


In [5]:
top_themes = theme_summary.nlargest(20, 'Сообществ').copy()
fig = px.scatter(
    top_themes,
    x='Каталог, %',
    y='Чекаут, %',
    size='Сообществ',
    color='Заказы',
    text='Тематика',
    hover_data={
        'Сообществ': ':,',
        'С_заказами': ':,',
        'Заказы': ':,',
        'GMV': ':,',
        'Каталог, %': ':.1f',
        'Чекаут, %': ':.1f',
    },
    color_continuous_scale='Blues',
    title='Тематики: размер базы и готовность к торговле',
    labels={'Каталог, %': 'Сообщества с каталогом, %', 'Чекаут, %': 'Сообщества с чекаутом, %'},
)
fig.update_traces(textposition='top center', marker={'line': {'width': 1, 'color': 'white'}})
fig.update_layout(template='plotly_white', height=650, margin=dict(l=70, r=40, t=90, b=70))
fig.show(config={'displaylogo': False, 'responsive': True})

**Вывод.** Внутри выборки есть два разных типа возможностей:

- крупные тематики с большим числом сообществ — хороший масштаб для массового онбординга, но часто с низкой готовностью;
- небольшие или средние тематики с высокой долей каталога и чекаута — хорошие кандидаты для пилотов и кейсов успеха.

Тематика сама по себе не доказывает потребность бизнеса. Она используется для отбора гипотез и должна дополняться интервью.

## 3. Сегментация по стадии развития

Сегменты построены по наблюдаемым признакам:

- **Контент без товарного контура** — нет каталога и нет заказов;
- **Каталог без чекаута** — товарный контур есть, но нет подключённого чекаута;
- **Чекаут без заказов** — чекаут подключён, но заказов нет;
- **Продавцы через чекаут** — есть хотя бы один заказ.

Сегментация показывает, какую задачу решать: подключение, активацию или рост эффективности.

In [6]:
groups['Сегмент развития'] = np.select(
    [
        groups['Есть заказы'],
        groups['Есть чекаут'] & ~groups['Есть заказы'],
        groups['Есть каталог'] & ~groups['Есть чекаут'],
    ],
    [
        'Продавцы через чекаут',
        'Чекаут без заказов',
        'Каталог без чекаута',
    ],
    default='Контент без товарного контура',
)
segment_summary = groups.groupby('Сегмент развития').agg(
    Сообществ=('ID бизнеса', 'size'),
    Доля=('ID бизнеса', lambda s: len(s) / len(groups) * 100),
    Медиана_подписчиков=('Число подписчиков', 'median'),
    Медиана_просмотров_товарного_контента=('Просмотры товарного контента', 'median'),
    Медиана_открытий_карточки=('Открытия карточки товара из контента', 'median'),
    Заказы=('Заказы через чекаут', 'sum'),
    GMV=('GMV через чекаут, ₽', 'sum'),
).sort_values('Сообществ', ascending=False)
segment_summary

,Сообществ,Доля,Медиана_подписчиков,Медиана_просмотров_товарного_контента,Медиана_открытий_карточки,Заказы,GMV
Сегмент развития,,,,,,,
Чекаут без заказов,5453,55.91,490.00,0.00,0.00,0,0
Контент без товарного контура,3589,36.80,420.00,0.00,0.00,0,0
Каталог без чекаута,414,4.24,"1,100.00",0.00,0.00,0,0
Продавцы через чекаут,298,3.06,"3,400.00",13.00,0.00,3478,9973027


In [7]:
segment_plot = segment_summary.reset_index()
fig = go.Figure(go.Bar(
    x=segment_plot['Сообществ'],
    y=segment_plot['Сегмент развития'],
    orientation='h',
    marker_color=['#B8C2CC', '#70A7C6', '#E07A5F', '#3978C6'],
    text=[f'{v:,.0f} ({p:.1f}%)' for v, p in zip(segment_plot['Сообществ'], segment_plot['Доля'])],
    textposition='outside',
    hovertemplate='%{y}<br>Сообществ: %{x:,.0f}<extra></extra>',
))
fig.update_layout(
    template='plotly_white',
    height=460,
    title='Воронка готовности бизнес-сообществ',
    xaxis_title='Количество сообществ',
    yaxis_title='',
    yaxis={'autorange': 'reversed'},
    margin=dict(l=180, r=100, t=80, b=60),
)
fig.show(config={'displaylogo': False, 'responsive': True})

### Что нужно сегментам

**Контент без товарного контура.** Цель — понять, какие товары или услуги можно упаковать в понятное предложение. Барьеры: отсутствие каталога, неясный следующий шаг после контента, недостаток товарного контента и слабая измеримость спроса. Помощь VK: шаблоны каталога, импорт ассортимента, рекомендации по карточке и публикации товара из поста или клипа.

**Каталог без чекаута.** Цель — сократить путь от интереса до заявки или заказа. Барьеры: подключение и настройка чекаута, доставка и оплата, страх ошибки и отсутствие доказательства спроса. Помощь VK: мастер настройки, диагностика обязательных полей, подсказки по доставке и оплате, безопасный пилот на части ассортимента.

**Чекаут без заказов.** Цель — активировать существующий инструмент. Барьеры: нет трафика к карточкам, слабое товарное предложение, неясная ценность кнопки, проблемы с ассортиментом или доверием. Помощь VK: рекомендации по контенту, поиску и карточке, раздельная аналитика показов → открытий → действия → оформления → оплаты.

**Продавцы через чекаут.** Цель — масштабировать заказы и повторные покупки. Барьеры: ограниченный охват, зависимость от отдельных источников, отсутствие повторных коммуникаций и неполная атрибуция. Помощь VK: клипы и реферальные механики, рекомендации ассортимента, удержание, тестирование продвижения и контроль качества заказа.

## 4. Подробный анализ целевых бизнесов

Для пилота выбираются не все продавцы, а сообщества, где уже виден спрос и есть возможность масштабирования. Для массового роста — крупные товарные сообщества без чекаута. Для доказательства ценности продукта — действующие продавцы с заказами и заметными открытиями карточки.

In [8]:
targets = groups[
    groups['Есть каталог']
    & (groups['Число подписчиков'] >= groups['Число подписчиков'].median())
].copy()
targets['Приоритетный сценарий'] = np.select(
    [
        targets['Есть заказы'] & targets['Открытия карточки товара из контента'].gt(0),
        ~targets['Есть чекаут'] & targets['Просмотры товарного контента'].gt(0),
        ~targets['Есть товарный контент'],
    ],
    [
        'Масштабирование продавцов',
        'Перевод спроса в чекаут',
        'Упаковка предложения и контента',
    ],
    default='Точечная диагностика',
)
target_summary = targets.groupby('Приоритетный сценарий').agg(
    Сообществ=('ID бизнеса', 'size'),
    Медиана_подписчиков=('Число подписчиков', 'median'),
    Медиана_открытий=('Открытия карточки товара из контента', 'median'),
    Сумма_заказов=('Заказы через чекаут', 'sum'),
    Сумма_GMV=('GMV через чекаут, ₽', 'sum'),
).sort_values('Сумма_заказов', ascending=False)
target_summary

,Сообществ,Медиана_подписчиков,Медиана_открытий,Сумма_заказов,Сумма_GMV
Приоритетный сценарий,,,,,
Упаковка предложения и контента,2188,"1,800.00",0.00,2232,5954504
Масштабирование продавцов,71,"5,800.00",9.00,1140,3295333
Точечная диагностика,98,"1,700.00",3.00,14,109740
Перевод спроса в чекаут,101,"1,900.00",0.00,2,27880


In [9]:
target_themes = targets.groupby('Тематика').agg(
    Сообществ=('ID бизнеса', 'size'),
    Медиана_подписчиков=('Число подписчиков', 'median'),
    Каталог=('Есть каталог', 'mean'),
    Чекаут=('Есть чекаут', 'mean'),
    Продавцы=('Есть заказы', 'sum'),
    Заказы=('Заказы через чекаут', 'sum'),
    GMV=('GMV через чекаут, ₽', 'sum'),
).reset_index()
target_themes['Каталог, %'] = target_themes['Каталог'] * 100
target_themes['Чекаут, %'] = target_themes['Чекаут'] * 100
target_themes = target_themes.nlargest(15, 'Сообществ')
fig = px.bar(
    target_themes.sort_values('Сообществ'),
    x='Сообществ',
    y='Тематика',
    orientation='h',
    color='Заказы',
    text='Продавцы',
    color_continuous_scale='Blues',
    hover_data={'Медиана_подписчиков': ':,', 'Каталог, %': ':.1f', 'Чекаут, %': ':.1f', 'GMV': ':,'},
    title='Приоритетные тематики среди товарных сообществ с медианным и выше охватом',
    labels={'Сообществ': 'Сообществ', 'Тематика': ''},
)
fig.update_traces(texttemplate='продавцов: %{text}', textposition='outside')
fig.update_layout(template='plotly_white', height=620, margin=dict(l=180, r=80, t=90, b=60))
fig.show(config={'displaylogo': False, 'responsive': True})

### Ответы на вопросы кейса

**Что нужно целевым бизнесам.** Товарным сообществам нужен короткий путь от контента к понятному предложению и заказу: каталог, карточка, наличие, цена, доставка, оплата и прозрачный статус результата. Действующим продавцам дополнительно нужны охват, повторные продажи и понимание, какой контент приводит к заказу.

**Каких целей они хотят достичь.** Получать продажи из уже существующей аудитории, снижать зависимость от внешних площадок и рекламы, превращать контент в измеримый коммерческий результат, а для сервисных бизнесов — получать заявки и диалоги.

**С какими трудностями сталкиваются.** Разный уровень цифровой готовности, отсутствие каталога или чекаута, слабый товарный контент, неочевидный следующий шаг, разрыв между открытием карточки и заказом, неполная аналитика внешних переходов и ограниченные ресурсы на регулярный контент.

**Какие задачи хотят решать во ВКонтакте.** Показать товар в ленте, клипах, сообществе и поиске; собрать ассортимент в каталог; принять заказ или заявку; подключить оплату и доставку; привлекать авторов и повторно возвращать покупателя.

**Что мешает расти.** Большая часть выборки находится до стадии транзакции. Даже наличие чекаута не гарантирует заказов. Для продавцов проблема смещается от подключения к качеству трафика, карточки и ассортимента.

**Как помочь развиваться.** Дать разные сценарии онбординга по сегментам, шаблоны товарного контента, мастер подключения каталога и чекаута, диагностику узкого места, рекомендации по источникам трафика, реферальные механики и когорную аналитику заказов.

## 5. Связь с остальными таблицами датасета

`Business_groups.csv` и таблицы Маркета отвечают на разные стороны кейса:

- `Business_groups.csv` показывает **предложение**: кто ведёт сообщества, что публикует и насколько готов к продаже;
- `Market_Partner_traffic.csv` показывает **спрос и входящий путь**: у Маркета важны витрина и «Другое», у партнёра — клипы и реферальные посты;
- `Market_Partner_categories.csv` показывает, в каких товарных категориях уже есть открытия и заказы;
- `Market_Partner_gender_age.csv` и региональная таблица описывают состав покупателей;
- `Market_Partner_dynamics.csv` показывает изменение событий по дням.

Прямого join по ID бизнеса или пользователю между таблицами нет. Поэтому нельзя утверждать, что конкретная тематика из `Business_groups.csv` породила конкретные заказы или что покупатели из одной таблицы принадлежат конкретным сообществам. Корректная связь — стратегическая: бизнесы создают предложение и контент, а Маркет даёт инфраструктуру обнаружения, карточки и оформления.

In [10]:
traffic_share = traffic[[f'Доля, %: Маркет Вконтакте', f'Доля, %: Партнёр 1']].copy()
traffic_share.columns = ['Маркет', 'Партнёр']
traffic_share = traffic_share.dropna(how='all').sort_values('Маркет', ascending=False)
fig = go.Figure()
fig.add_bar(y=traffic_share.index, x=traffic_share['Маркет'], name='Маркет', orientation='h', marker_color='#3978C6')
fig.add_bar(y=traffic_share.index, x=traffic_share['Партнёр'], name='Партнёр', orientation='h', marker_color='#F2994A')
fig.update_layout(
    template='plotly_white',
    barmode='group',
    height=500,
    title='Источники открытий карточек: спрос со стороны пользователей',
    xaxis_title='Доля открытий, %',
    yaxis_title='Источник',
    legend=dict(orientation='h', y=1.08),
    margin=dict(l=170, r=30, t=100, b=60),
)
fig.show(config={'displaylogo': False, 'responsive': True})

## 6. Рекомендации и KPI пилота

**Приоритет 1 — продавцы через чекаут.** Использовать как контрольную группу для экспериментов с клипами, реферальными постами, ассортиментом и повторными продажами.

**Приоритет 2 — товарные сообщества без чекаута.** Это основной резерв роста числа бизнесов с продажами внутри ВК. Цель — довести их до первого оформленного заказа через упрощённый онбординг.

**Приоритет 3 — крупные сообщества без товарного контура.** Не навязывать чекаут; сначала подтвердить товарную потребность и предложить каталог либо сценарий заявок.

KPI следует разделять по этапам: активные сообщества с каталогом; товарные показы; открытия карточек; нажатия с разметкой назначения; переходы к оформлению; оформленные и оплаченные заказы; GMV; повторные заказы; стоимость привлечения и доля заказов, атрибутированных контенту. Нажатие на кнопку нельзя считать чекаутом без отдельного события назначения и оформления.

## Ограничения

- Данные синтетические и относятся к одному месяцу.
- `Автор оригинального контента` и часть ценовых полей имеют пропуски; нули в финансовых полях могут означать отсутствие продаж, а не отсутствие измерения.
- Тематика и тип бизнеса — классификации, а не подтверждённые интервью потребности.
- Не измерены прибыль, возвраты, отмены, качество заказов и причинность между рекламой, контентом и продажами.
- Для сервисных сообществ товарные метрики не являются корректным KPI; им нужен отдельный сценарий лидов и переписки.